# Predict the star a person would give

A person has rated some movies. The question is the rating they would give a movie they have not rated yet. That missing number is what a recommender is trying to fill in.

This path uses the MovieLens 100K tables collected by GroupLens at the University of Minnesota: 100000 ratings, on a scale from 1 to 5, from 943 users, on 1682 movies. The ratings were gathered from 19 September 1997 through 22 April 1998. The conditions of use are in `data/README`. Publications that use the tables cite Harper and Konstan, 2015, *The MovieLens Datasets: History and Context*. Nothing in these lessons is an endorsement by the University of Minnesota or by GroupLens.

The error used here is the root mean squared error. For ratings $r_i$ and predictions $\hat r_i$,

$$
\mathrm{RMSE}
= \sqrt{\dfrac{1}{n}\sum_{i=1}^{n}(\hat r_i - r_i)^2}.
$$

Try three ratings, $1$, $3$, and $5$, all predicted as their mean $3$. The errors are $-2$, $0$, and $2$.

$$
\dfrac{(-2)^2 + 0^2 + 2^2}{3} = \dfrac{8}{3},
\qquad
\mathrm{RMSE} = \sqrt{\dfrac{8}{3}}.
$$

Predicting $4$ instead gives squared errors $9$, $1$, and $1$, and the mean of those squares is $11/3$. Since $11/3 > 8/3$, the constant $4$ is a worse squared-error prediction than the mean. Among constant predictions, the mean is the one that minimizes this error.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "16-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# Three ratings, two constant predictions, one exact RMSE.
import math
from fractions import Fraction
ratings_toy = [1, 3, 5]
mean = Fraction(sum(ratings_toy), 3)
squared = [(mean - rating) ** 2 for rating in ratings_toy]
mean_square = sum(squared) / 3
other = [Fraction((4 - rating) ** 2) for rating in ratings_toy]
other_mean = sum(other) / 3
rmse = math.sqrt(float(mean_square))
print(mean, mean_square, other_mean, rmse)
assert mean == 3
assert mean_square == Fraction(8, 3)
assert other_mean == Fraction(11, 3)
assert other_mean > mean_square
assert abs(rmse - math.sqrt(8 / 3)) < 1e-12
assert len(ratings) == 100000


3 8/3 11/3 1.632993161855452


## Pitfall

$\sqrt{8/3}$ is the error of one constant on three toy ratings. It is not a score on MovieLens. A score has to name which ratings were hidden before the prediction was built.

## Summary

The question is a missing rating from 1 to 5. RMSE is the square root of the mean squared gap. On $1$, $3$, and $5$, the mean $3$ has mean squared error $8/3$, and the constant $4$ has $11/3$.
